# LeetCode #179 — Largest Number: sắp xếp với phép so sánh tự định nghĩa `a + b` vs `b + a`

**Mức độ:** Medium · **Chủ đề:** Array, String, Greedy, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def largestNumber(self, nums: List[int]) -> str:
```

Mục tiêu của notebook:

1. Vì sao sắp xếp **theo số** hay **theo chuỗi** đều **sai** (`[3, 30]`: `"330"` lớn hơn `"303"`).
2. Quy tắc tham lam: đặt `a` trước `b` nếu chuỗi ghép `a + b > b + a`.
3. Dùng `functools.cmp_to_key` để sắp xếp bằng hàm so sánh; hiểu vì sao quy tắc này là một **thứ tự hợp lệ** (bắc cầu).
4. Trường hợp biên: toàn số 0 → `"0"`, không phải `"000"`.

# 1. Bài toán là gì?

Cho danh sách số nguyên không âm `nums`, sắp xếp lại chúng để khi **ghép liền** các số thì được số **lớn nhất**. Vì kết quả có thể rất lớn, trả về dạng **chuỗi**.

**Constraints (theo đề):**

- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 10^9`

---

## Ví dụ

```text
[10, 2]               ->  "210"
[3, 30, 34, 5, 9]     ->  "9534330"
[0, 0]                ->  "0"
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp theo tiêu chí tự định nghĩa (key và comparator)
2. Chuỗi (string) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp theo tiêu chí tự định nghĩa (key và comparator)

### `key=`: biến mỗi phần tử thành "khóa" để so sánh

`sorted(a, key=f)` sắp theo `f(x)`. Khóa có thể là **tuple** để sắp nhiều tiêu chí: so tiêu chí đầu, bằng nhau thì so tiêu chí sau.

In [1]:
people = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print(sorted(people, key=lambda p: p[1]))                  # theo tuổi; ổn định: giữ thứ tự cũ khi bằng
print(sorted(people, key=lambda p: (-p[1], p[0])))         # tuổi giảm dần, rồi tên tăng dần

[('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
[('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]


### Sắp xếp **ổn định** (stable)

Python dùng Timsort — **ổn định**: hai phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu. Nhờ vậy có thể sắp nhiều lần, tiêu chí **phụ trước**, tiêu chí **chính sau**.

### Comparator: khi không viết được thành một khóa đơn giản

Có những thứ tự định nghĩa bằng **so sánh từng cặp** — ví dụ "đặt `a` trước `b` nếu `a + b > b + a`" (ghép chuỗi). Dùng `functools.cmp_to_key(cmp)`, trong đó `cmp(a, b)` trả về **số âm** nếu `a` đứng trước, **dương** nếu `b` đứng trước, `0` nếu như nhau.

In [2]:
from functools import cmp_to_key

def cmp_concat(a, b):
    if a + b > b + a:
        return -1
    if a + b < b + a:
        return 1
    return 0

parts = ["3", "30", "34", "5", "9"]
print("sắp theo chữ giảm dần:", sorted(parts, reverse=True), "→", "".join(sorted(parts, reverse=True)))
print("sắp theo comparator   :", sorted(parts, key=cmp_to_key(cmp_concat)), "→", "".join(sorted(parts, key=cmp_to_key(cmp_concat))))

sắp theo chữ giảm dần: ['9', '5', '34', '30', '3'] → 9534303
sắp theo comparator   : ['9', '5', '34', '3', '30'] → 9534330


### Comparator phải là một **thứ tự hợp lệ**

Để thuật toán sắp xếp đúng, quan hệ "đứng trước" phải **bắc cầu**: `a` trước `b` và `b` trước `c` ⟹ `a` trước `c`. Comparator tùy tiện (không bắc cầu) cho kết quả phụ thuộc thứ tự đầu vào — không có "thứ tự đúng" nào cả.

In [3]:
def cmp_bad(a, b):                 # kéo-búa-bao: không bắc cầu
    beats = {("keo", "giay"), ("giay", "bua"), ("bua", "keo")}
    return -1 if (a, b) in beats else (1 if (b, a) in beats else 0)

print(sorted(["keo", "giay", "bua"], key=cmp_to_key(cmp_bad)))
print(sorted(["bua", "keo", "giay"], key=cmp_to_key(cmp_bad)))

['keo', 'giay', 'bua']
['bua', 'keo', 'giay']


### Lỗi hay gặp

- Comparator trả về `True/False` thay vì số âm/dương/0 → `cmp_to_key` hiểu sai.
- Quên rằng so sánh chuỗi là **theo từ điển**: `"9" > "10"`.
- Comparator không bắc cầu → kết quả "ngẫu nhiên".

## 2.2 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [4]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [5]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [6]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [7]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.3 Áp dụng vào bài này

Câu hỏi "số nào đứng trước" không viết được thành một khóa đơn giản (sắp theo chữ cho `"3" > "30"` nhưng `"330" > "303"`), nên ta **định nghĩa phép so sánh**: `a` trước `b` nếu `a + b > b + a` (ghép **chuỗi**), dùng `cmp_to_key`. Phép so sánh này **bắc cầu** nên sắp xếp hợp lệ. Cuối cùng xử lý trường hợp toàn số 0 (`"00"` → `"0"`).

# 3. Nền tảng

In [8]:
from typing import List
from functools import cmp_to_key
from itertools import permutations
import random, time

print("sắp theo số giảm dần:", sorted([3, 30, 34, 5, 9], reverse=True), "→", "".join(map(str, sorted([3, 30, 34, 5, 9], reverse=True))))
print("sắp theo chuỗi giảm dần:", sorted(["3", "30", "34", "5", "9"], reverse=True), "→", "".join(sorted(["3", "30", "34", "5", "9"], reverse=True)))

sắp theo số giảm dần: [34, 30, 9, 5, 3] → 3430953
sắp theo chuỗi giảm dần: ['9', '5', '34', '30', '3'] → 9534303


- Theo **số**: `34 > 30 > 9 …` → `"34309…"` — sai hoàn toàn (số có nhiều chữ số chưa chắc nên đứng trước).
- Theo **chuỗi**: gần đúng (`"9" > "5" > "34"…`) nhưng `"30" > "3"` (chuỗi dài hơn có tiền tố chung được coi là lớn hơn) → `"…303"` trong khi `"…330"` lớn hơn.

## Hàm so sánh và `cmp_to_key`

`sorted(..., key=...)` cần một hàm **khóa**. Khi tiêu chí là "so sánh **hai** phần tử với nhau", ta viết hàm `cmp(a, b)` trả về số **âm** nếu `a` đứng trước, **dương** nếu `b` đứng trước, `0` nếu như nhau — rồi đổi thành khóa bằng `functools.cmp_to_key`.

In [9]:
def cmp_len(a, b):            # ví dụ: sắp chuỗi theo độ dài, ngắn trước
    return len(a) - len(b)

print(sorted(["ccc", "a", "bb"], key=cmp_to_key(cmp_len)))

['a', 'bb', 'ccc']


# 4. Baseline: thử mọi hoán vị

`n!` cách sắp xếp — chỉ dùng được với `n` rất nhỏ, nhưng là đáp án đối chiếu tốt.

In [10]:
def largest_brute(nums: List[int]) -> str:
    best = max("".join(p) for p in permutations(map(str, nums)))
    return "0" if best[0] == "0" else best

print(largest_brute([10, 2]), largest_brute([3, 30, 34, 5, 9]), largest_brute([0, 0]))

210 9534330 0


`max` của các chuỗi **cùng độ dài** (mọi hoán vị có cùng tổng số chữ số) theo thứ tự từ điển cũng chính là số lớn nhất.

# 5. Ý tưởng chính: so sánh `a + b` với `b + a`

Với hai số (dạng chuỗi) `a` và `b`, chỉ có hai cách đặt cạnh nhau: `a + b` hoặc `b + a`. Hai chuỗi này **cùng độ dài**, so được trực tiếp. Quy tắc tham lam:

```text
a đứng trước b  ⟺  a + b > b + a
```

```text
a = "3", b = "30":   "330" > "303"  → 3 trước 30
a = "9", b = "34":   "934" > "349"  → 9 trước 34
a = "34", b = "3":   "343" > "334"  → 34 trước 3
```

Sắp xếp toàn bộ danh sách theo quy tắc này rồi ghép lại. Nếu chuỗi bắt đầu bằng `"0"` thì mọi số đều là 0 → trả `"0"`.

In [11]:
def compare(a: str, b: str) -> int:
    if a + b > b + a:
        return -1          # a đứng trước
    if a + b < b + a:
        return 1           # b đứng trước
    return 0

def largest_number(nums: List[int]) -> str:
    strs = sorted(map(str, nums), key=cmp_to_key(compare))
    res = "".join(strs)
    return "0" if res[0] == "0" else res

print(largest_number([10, 2]), largest_number([3, 30, 34, 5, 9]), largest_number([0, 0]))

210 9534330 0


# 6. Mô phỏng: các phép so sánh mà sắp xếp đã thực hiện

In [12]:
def trace_largest(nums):
    log = []
    def cmp_logged(a, b):
        r = compare(a, b)
        log.append(f"   {a!r:>5} vs {b!r:<5}: {a + b} {'>' if r < 0 else '<' if r > 0 else '='} {b + a} → {a if r <= 0 else b} trước")
        return r
    strs = sorted(map(str, nums), key=cmp_to_key(cmp_logged))
    print("\n".join(log))
    print("thứ tự:", strs, "→", "".join(strs))

trace_largest([3, 30, 34, 5, 9])

    '30' vs '3'  : 303 < 330 → 3 trước
    '34' vs '30' : 3430 > 3034 → 34 trước
     '3' vs '30' : 330 > 303 → 3 trước
    '34' vs '30' : 3430 > 3034 → 34 trước
    '34' vs '3'  : 343 > 334 → 34 trước
     '5' vs '3'  : 53 > 35 → 5 trước
     '5' vs '34' : 534 > 345 → 5 trước
     '9' vs '3'  : 93 > 39 → 9 trước
     '9' vs '34' : 934 > 349 → 9 trước
     '9' vs '5'  : 95 > 59 → 9 trước


thứ tự: ['9', '5', '34', '3', '30'] → 9534330


# 7. Phiên bản đúng format LeetCode

In [13]:
class Solution:
    def largestNumber(self, nums: List[int]) -> str:
        strs = sorted(map(str, nums), key=cmp_to_key(lambda a, b: (b + a > a + b) - (b + a < a + b)))
        res = "".join(strs)
        return "0" if res[0] == "0" else res

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.largestNumber([10, 2]), "210")
check("Ví dụ 2", sol.largestNumber([3, 30, 34, 5, 9]), "9534330")
check("Toàn 0", sol.largestNumber([0, 0]), "0")
check("Một số", sol.largestNumber([1]), "1")
check("Tiền tố", sol.largestNumber([121, 12]), "12121")
check("Số lớn", sol.largestNumber([10**9, 999999999]), "9999999991000000000")

random.seed(179)
for _ in range(3000):
    nums = [random.choice([0, random.randint(0, 9), random.randint(0, 99), random.randint(0, 999)]) for _ in range(random.randint(1, 6))]
    assert sol.largestNumber(nums) == largest_brute(nums) == largest_number(nums), nums
print("3000 bộ ngẫu nhiên: khớp brute force (thử mọi hoán vị)")

big = [random.randint(0, 10**9) for _ in range(100)]
t0 = time.perf_counter(); sol.largestNumber(big); t1 = time.perf_counter()
print(f"n = 100: {(t1 - t0) * 1000:.2f} ms")

Ví dụ 1        PASS  210
Ví dụ 2        PASS  9534330
Toàn 0         PASS  0
Một số         PASS  1
Tiền tố        PASS  12121
Số lớn         PASS  9999999991000000000


3000 bộ ngẫu nhiên: khớp brute force (thử mọi hoán vị)
n = 100: 0.16 ms


# 8. Vì sao đúng? Độ phức tạp?

**(1) Quy tắc là một thứ tự hợp lệ.** Coi chuỗi `a` dài `p` chữ số như số `A`. Ghép `a + b` có giá trị `A · 10^q + B` (`q` = độ dài `b`). Vậy

```text
a + b > b + a  ⟺  A·10^q + B > B·10^p + A  ⟺  A(10^q − 1) > B(10^p − 1)  ⟺  A/(10^p − 1) > B/(10^q − 1)
```

Mỗi chuỗi tương ứng một số thực `f(a) = A / (10^p − 1)` (với `a = "0"` thì `f = 0`), và quy tắc chỉ là **so sánh `f`** → có tính bắc cầu, sắp xếp được.

**(2) Thứ tự đó cho số lớn nhất** (lập luận trao đổi): trong một cách sắp xếp bất kỳ, nếu có hai phần tử **kề nhau** `b a` mà `a + b > b + a`, đổi chỗ chúng thành `a b` làm số **lớn hơn** (phần trước và sau không đổi, đoạn giữa tăng). Cách sắp tối ưu không thể có cặp kề như vậy → nó được sắp theo quy tắc.

```text
Time:  O(n log n) phép so sánh, mỗi phép O(k) với k = số chữ số (≤ 10) → O(n log n · k)
Space: O(n · k)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Sắp xếp theo chuỗi giảm dần

In [15]:
def wrong_string_sort(nums):
    return "".join(sorted(map(str, nums), reverse=True))

for t in [[3, 30, 34, 5, 9], [12, 121]]:
    print(t, "→ sai:", wrong_string_sort(t), "| đúng:", sol.largestNumber(t))

[3, 30, 34, 5, 9] → sai: 9534303 | đúng: 9534330
[12, 121] → sai: 12112 | đúng: 12121


## Lỗi 2: Quên trường hợp toàn số 0

In [16]:
def wrong_zeros(nums):
    return "".join(sorted(map(str, nums), key=cmp_to_key(compare)))

print("Sai :", wrong_zeros([0, 0, 0]), "| Đúng:", sol.largestNumber([0, 0, 0]))

Sai : 000 | Đúng: 0


## Lỗi 3: "Kéo dài" chuỗi bằng cách lặp chữ số cuối để so sánh

Một mẹo hay gặp: so `a` và `b` sau khi kéo dài chúng bằng **chữ số cuối** tới cùng độ dài. Không đúng tổng quát:

In [17]:
def wrong_pad_last(nums):
    L = 11
    strs = sorted(map(str, nums), key=lambda s: s + s[-1] * (L - len(s)), reverse=True)
    res = "".join(strs)
    return "0" if res[0] == "0" else res

random.seed(3)
for _ in range(20000):
    t = [random.randint(0, 999) for _ in range(random.randint(2, 4))]
    if wrong_pad_last(t) != sol.largestNumber(t):
        print("Phản ví dụ:", t, "| sai:", wrong_pad_last(t), "| đúng:", sol.largestNumber(t))
        break

Phản ví dụ: [814, 81, 580] | sai: 81481580 | đúng: 81814580


Kéo dài bằng cách **lặp lại cả chuỗi** (`(s * 11)[:11]`, vì `s` lặp vô hạn chính là `f(s)` ở mục 8) thì đúng — một khóa sắp xếp hợp lệ không cần `cmp_to_key`:

In [18]:
def largest_by_repeat_key(nums):
    strs = sorted(map(str, nums), key=lambda s: (s * 11)[:11], reverse=True)
    res = "".join(strs)
    return "0" if res[0] == "0" else res

for _ in range(3000):
    t = [random.randint(0, 999) for _ in range(random.randint(1, 5))]
    assert largest_by_repeat_key(t) == sol.largestNumber(t)
print("khóa (s * 11)[:11]: khớp lời giải chính trên 3000 bộ")

khóa (s * 11)[:11]: khớp lời giải chính trên 3000 bộ


## Lỗi 4: Hàm so sánh trả về `True/False`

`cmp_to_key` cần **số âm / 0 / dương**. Trả `a + b > b + a` (bool) nghĩa là chỉ có `0` và `1` — không bao giờ "âm" → thứ tự sai.

In [19]:
def wrong_bool_cmp(nums):
    strs = sorted(map(str, nums), key=cmp_to_key(lambda a, b: a + b > b + a))
    return "".join(strs)

print("Sai :", wrong_bool_cmp([3, 30, 34, 5, 9]), "| Đúng:", sol.largestNumber([3, 30, 34, 5, 9]))

Sai : 3303459 | Đúng: 9534330


# 10. Mẹo nhận diện pattern

- **Sắp xếp theo "ghép/đặt cạnh nhau thì tốt hơn"** → so `a + b` với `b + a` (hoặc tổng quát: so hai thứ tự của **một cặp**).
- **Chứng minh tham lam bằng hoán đổi hai phần tử kề nhau** (exchange argument).
- Python: `functools.cmp_to_key` khi tiêu chí không biểu diễn được bằng một khóa đơn.

Bài liên quan: **#2165 Smallest Value of the Rearranged Number**, **#1029 Two City Scheduling** (sắp theo hiệu), **#406 Queue Reconstruction by Height**, **#Offer 45** (số **nhỏ** nhất ghép được), **#1665**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Sắp xếp `[824, 8247]` theo quy tắc, và `[0, 9, 90, 909]`.

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_largest_number(nums: List[int]) -> str:
    def cmp(a: str, b: str) -> int:
        # TODO: trả -1 nếu a đứng trước, 1 nếu b đứng trước, 0 nếu như nhau
        pass
    strs = sorted(map(str, nums), key=cmp_to_key(cmp))
    res = "".join(strs)
    return "0" if res[0] == "0" else res


# assert exercise_2_largest_number([10, 2]) == "210"
# assert exercise_2_largest_number([3, 30, 34, 5, 9]) == "9534330"
# assert exercise_2_largest_number([0, 0]) == "0"

## Bài 3 — Biến thể: số **nhỏ nhất** ghép được (giữ số 0 ở đầu)

`[3, 30, 34, 5, 9]` → `"3033459"`; `[0, 1]` → `"01"`.

In [21]:
def exercise_3_smallest_number(nums: List[int]) -> str:
    # TODO
    pass


# assert exercise_3_smallest_number([3, 30, 34, 5, 9]) == "3033459"
# assert exercise_3_smallest_number([10, 2]) == "102"
# assert exercise_3_smallest_number([0, 1]) == "01"

## Bài 4 — Biến thể: số lớn nhất chia hết cho 3 ghép từ **một số** các phần tử

Được bỏ bớt phần tử (giữ ít nhất một). Gợi ý: tổng chữ số chia hết cho 3 ⟺ số chia hết cho 3; bỏ **ít nhất** các phần tử để tổng `mod 3 = 0`, ưu tiên bỏ phần tử "ít đóng góp" — nhưng cách an toàn với `n` nhỏ là thử mọi tập con. Trả `""` nếu không có.

In [22]:
def exercise_4_largest_div3(nums: List[int]) -> str:
    # TODO (n <= 12)
    pass


# assert exercise_4_largest_div3([8, 1, 9]) == "981"
# assert exercise_4_largest_div3([1]) == ""
# assert exercise_4_largest_div3([0, 0, 2]) == "0"

## Bài 5 — Mini challenge: khóa sắp xếp không cần `cmp_to_key`

Viết `largest` dùng `key=` trả về **phân số** `A / (10^p − 1)` (mục 8) bằng `fractions.Fraction` để so sánh chính xác.

In [23]:
from fractions import Fraction

def exercise_5_largest_by_fraction(nums: List[int]) -> str:
    # TODO
    pass


# assert exercise_5_largest_by_fraction([3, 30, 34, 5, 9]) == "9534330"
# assert exercise_5_largest_by_fraction([0, 0]) == "0"
# assert exercise_5_largest_by_fraction([121, 12]) == "12121"

# 12. Gợi ý

**Bài 1**
- Mức 1: so `"8248247"` với `"8247824"`; chạy `trace_largest` sau khi tự làm.

**Bài 3**
- Mức 1: đảo quy tắc: `a` trước `b` nếu `a + b < b + a`. Không cần xử lý số 0 đầu.

**Bài 4**
- Mức 1: duyệt mọi tập con khác rỗng có tổng chữ số `% 3 == 0`, với mỗi tập tính `largest_number`, giữ chuỗi lớn nhất (so độ dài trước, sau đó so chuỗi; nhớ `"0"` đặc biệt).

**Bài 5**
- Mức 1: `key=lambda s: Fraction(int(s), 10 ** len(s) - 1)`, `reverse=True`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
trace_largest([824, 8247])
trace_largest([0, 9, 90, 909])

   '8247' vs '824': 8247824 < 8248247 → 824 trước
thứ tự: ['824', '8247'] → 8248247
     '9' vs '0'  : 90 > 09 → 9 trước
    '90' vs '9'  : 909 < 990 → 9 trước
     '9' vs '90' : 990 > 909 → 9 trước
    '90' vs '0'  : 900 > 090 → 90 trước
    '90' vs '0'  : 900 > 090 → 90 trước
    '90' vs '9'  : 909 < 990 → 9 trước
   '909' vs '90' : 90990 > 90909 → 909 trước
   '909' vs '9'  : 9099 < 9909 → 9 trước
thứ tự: ['9', '909', '90', '0'] → 9909900


## Đáp án Bài 2

In [25]:
def answer_2_largest_number(nums: List[int]) -> str:
    return largest_number(nums)

assert answer_2_largest_number([10, 2]) == "210"
assert answer_2_largest_number([3, 30, 34, 5, 9]) == "9534330"
assert answer_2_largest_number([0, 0]) == "0"
print("Bài 2 PASS (giống largest_number ở mục 5)")

Bài 2 PASS (giống largest_number ở mục 5)


## Đáp án Bài 3

In [26]:
def answer_3_smallest_number(nums: List[int]) -> str:
    return "".join(sorted(map(str, nums), key=cmp_to_key(lambda a, b: compare(b, a))))

assert answer_3_smallest_number([3, 30, 34, 5, 9]) == "3033459"
assert answer_3_smallest_number([10, 2]) == "102"
assert answer_3_smallest_number([0, 1]) == "01"
for _ in range(2000):
    t = [random.randint(0, 99) for _ in range(random.randint(1, 5))]
    assert answer_3_smallest_number(t) == min("".join(p) for p in permutations(map(str, t)))
print("Bài 3 PASS")

Bài 3 PASS

## Đáp án Bài 4

In [27]:
def num_key(s: str):
    s = s.lstrip("0") or "0"
    return (len(s), s)

def answer_4_largest_div3(nums: List[int]) -> str:
    best = ""
    n = len(nums)
    for mask in range(1, 1 << n):
        pick = [nums[i] for i in range(n) if mask >> i & 1]
        if sum(sum(map(int, str(x))) for x in pick) % 3 == 0:
            cand = largest_number(pick)
            if best == "" or num_key(cand) > num_key(best):
                best = cand
    return best

assert answer_4_largest_div3([8, 1, 9]) == "981"
assert answer_4_largest_div3([1]) == ""
assert answer_4_largest_div3([0, 0, 2]) == "0"
for _ in range(500):
    t = [random.randint(0, 30) for _ in range(random.randint(1, 5))]
    r = answer_4_largest_div3(t)
    assert r == "" or int(r) % 3 == 0
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_largest_by_fraction(nums: List[int]) -> str:
    strs = sorted(map(str, nums), key=lambda s: Fraction(int(s), 10 ** len(s) - 1), reverse=True)
    res = "".join(strs)
    return "0" if res[0] == "0" else res

assert answer_5_largest_by_fraction([3, 30, 34, 5, 9]) == "9534330"
assert answer_5_largest_by_fraction([0, 0]) == "0"
assert answer_5_largest_by_fraction([121, 12]) == "12121"
for _ in range(3000):
    t = [random.randint(0, 10**4) for _ in range(random.randint(1, 6))]
    assert answer_5_largest_by_fraction(t) == sol.largestNumber(t)
print("Bài 5 PASS (khớp lời giải chính trên 3000 bộ)")

Bài 5 PASS (khớp lời giải chính trên 3000 bộ)


# 14. Tổng kết

> Đổi sang chuỗi, sắp xếp với quy tắc "`a` trước `b` ⟺ `a + b > b + a`" (`functools.cmp_to_key`), ghép lại; nếu bắt đầu bằng `"0"` thì trả `"0"`. Quy tắc đúng vì nó tương đương so sánh `A / (10^p − 1)` (bắc cầu) và đổi chỗ cặp kề sai thứ tự luôn làm số lớn hơn.

---

## Checklist trước khi nộp

- [ ] So `a + b` với `b + a` (không so số, không so chuỗi trực tiếp)?
- [ ] Hàm so sánh trả âm / 0 / dương (không trả bool)?
- [ ] Toàn số 0 → `"0"`?
- [ ] Trả về **chuỗi**?